In [ ]:
import pandas as pd
import numpy as np
import hopsworks
import warnings
warnings.filterwarnings("ignore")

from evidently import ColumnMapping

from evidently.report import Report
from evidently.metrics.base_metric import generate_column_metrics
from evidently.metrics import *

from evidently.test_suite import TestSuite
from evidently.tests import *
from ml_pipeline.utils import *
from ml_pipeline.data_prep import processing_and_feature_engg

import wandb
import os
import joblib
import xgboost as xgb

In [ ]:
# Load configuration from config.yaml
config = load_config()

In [ ]:
routes_df = fetch_data(config, 'routes_details_fg')
trucks_df = fetch_data(config, 'truck_details_fg')
drivers_df = fetch_data(config, 'drivers_details_fg')
truck_schedule_df = fetch_data(config, 'truck_schedule_details_fg')
traffic_df = fetch_data(config, 'traffic_details_fg')
city_weather_df = fetch_data(config, 'city_weather_details_fg')
route_weather_df = fetch_data(config, 'route_weather_details_fg')


In [ ]:
# Assuming you have a 'timestamp' column in each dataframe

# Define a cutoff timestamp for separating "new" and "old" data
cutoff_date = '2019-02-01'  # Adjust the timestamp as needed

# Repeat the process for other dataframes

# Separate truck_schedule_df into "new" and "old"
new_truck_schedule_df = truck_schedule_df[truck_schedule_df['departure_date'] >= cutoff_date]
old_truck_schedule_df = truck_schedule_df[truck_schedule_df['departure_date'] < cutoff_date]

# Separate traffic_df into "new" and "old"
new_traffic_df = traffic_df[traffic_df['date'] >= cutoff_date]
old_traffic_df = traffic_df[traffic_df['date'] < cutoff_date]

# Separate city_weather_df into "new" and "old"
new_city_weather_df = city_weather_df[city_weather_df['date'] >= cutoff_date]
old_city_weather_df = city_weather_df[city_weather_df['date'] < cutoff_date]

# Separate route_weather_df into "new" and "old"
new_route_weather_df = route_weather_df[route_weather_df['date'] >= cutoff_date]
old_route_weather_df = route_weather_df[route_weather_df['date'] < cutoff_date]

In [ ]:
# data constraint check for all streaming data

def run_data_quality_check(data_name, reference_data, current_data):

    tests = TestSuite(tests=[
    TestNumberOfRowsWithMissingValues(),
    TestNumberOfDuplicatedRows(),
    TestNumberOfDuplicatedColumns(),
    TestColumnsType()
    ])

    tests.run(reference_data=reference_data, current_data=current_data)
    # tests.save_html(data_name+"_results.html")
    tests.show(mode='inline')

    return tests, tests.json()

In [ ]:
routes_weather_quality_dashboard, routes_weather_quality_results = run_data_quality_check('Route_Weather_Streaming',route_weather_df, new_route_weather_df)
routes_weather_quality_dashboard.show(mode='inline')

In [ ]:
traffic_quality_dashboard, traffic_quality_results = run_data_quality_check('Traffic_Streaming',traffic_df, new_traffic_df)
traffic_quality_dashboard.show(mode='inline')

In [ ]:
new_truck_schedule_df['delay'] = new_truck_schedule_df['delay'].astype("int64")

In [ ]:
truck_schedule_quality_dashboard, truck_schedule_quality_results = run_data_quality_check('Schedule_Streaming', truck_schedule_df, new_truck_schedule_df)
truck_schedule_quality_dashboard.show(mode='inline')

In [ ]:
city_weather_quality_dashboard, city_weather_quality_results = run_data_quality_check('City_Weather_Sttreaming',city_weather_df, new_city_weather_df)
city_weather_quality_dashboard

In [ ]:
final_merge_new = processing_and_feature_engg(routes_df, route_weather_df, drivers_df, trucks_df, traffic_df, new_truck_schedule_df, city_weather_df, 0)

# Reference (Training Data)

In [ ]:
unnecessary_cols = ['unique_id', 'truck_id', 'route_id', 'departure_date',
       'estimated_arrival', 'route_description',
       'estimated_arrival_nearest_hour', 'departure_date_nearest_hour',
       'origin_id', 'destination_id', 'origin_description',
       'destination_description',  'driver_id', 'name', 'gender', 'fuel_type',
       'driving_style',  'vehicle_no']

In [ ]:
final_merge_new.drop(columns=unnecessary_cols).columns

In [ ]:
final_merge = fetch_data(config, 'final_data')

In [ ]:
final_merge_dashboard, final_merge_results = run_data_quality_check('Final_merge',final_merge, final_merge_new)
final_merge_dashboard.show(mode='inline')

# Data Drift Report

In [ ]:
target = 'Delay'
prediction = 'Delay_Predictions'
numerical_features =  ['route_avg_temp', 'route_avg_wind_speed',
        'route_avg_precip', 'route_avg_humidity', 'route_avg_visibility',
        'route_avg_pressure', 'distance', 'average_hours',
        'origin_temp', 'origin_wind_speed', 'origin_precip', 'origin_humidity',
         'origin_pressure',
        'destination_temp','destination_wind_speed','destination_precip',
        'destination_humidity', 'destination_pressure',
          'avg_no_of_vehicles', 'truck_age','load_capacity_pounds', 'mileage_mpg',
          'age', 'experience','average_speed_mph']
categorical_features = ['origin_visibility','destination_visibility', 'accident',  'ratings','is_midnight']

In [ ]:
column_mapping = ColumnMapping()

column_mapping.target = target
column_mapping.prediction = prediction
column_mapping.numerical_features = numerical_features
column_mapping.categorical_features = categorical_features

In [ ]:
#dataset-level metrics
# remove unnecessary columns and then check drift or check drift on training data set after preprocessing?


data_drift_dataset_report = Report(metrics=[

    DatasetDriftMetric(drift_share=0.1),
    # can give separate thresholds in dic, or sep for cat and numerical variables as well in data drift table
    DataDriftTable(),

])

data_drift_dataset_report.run(reference_data=final_merge.drop(columns=unnecessary_cols), current_data=final_merge_new.drop(columns=unnecessary_cols),column_mapping = column_mapping)

data_drift_dataset_report.show(mode='inline')

# Model Drift and Reference Data Predictions

In [ ]:
truck_data_encoder = joblib.load('output/truck_data_encoder.pkl')

truck_data_scaler = joblib.load('output/truck_data_scaler.pkl')

In [ ]:
model_dir = fetch_best_model(config)

dir_list = os.listdir(model_dir)

truck_model = joblib.load(model_dir+'/'+dir_list[0])

In [ ]:
X_train, y_train = processing_new_data(config, final_merge, truck_data_scaler, truck_data_encoder)

In [ ]:
X_train.columns

In [ ]:
def predict_delay(truck_model, X_data, y_data, merged_data):
    """
    Predict delay using the given truck model and data.

    Parameters:
    - truck_model (xgb.core.Booster or other model): The trained model for predicting delay.
    - X_data (DataFrame): Features used for input
    - y_data (Series): Target variable
    - merged_data (DataFrame): Merged DataFrame containing truck_id and route_id.

    Returns:
    - DataFrame: DataFrame with delay predictions and additional columns for truck_id, route_id, and actual delay.
    """
    try:
        if isinstance(truck_model, xgb.core.Booster):
            dtrain = xgb.DMatrix(X_data, label=y_data)
            y_pred = truck_model.predict(dtrain)
        else:
            y_pred = truck_model.predict(X_data)

        X_data['Delay_Predictions'] = y_pred
        X_data['truck_id'] = merged_data['truck_id']
        X_data['route_id'] = merged_data['route_id']

        X_data['Delay'] = y_data
    except Exception as e:
        print(f"An error occurred during prediction {e}")

    return X_data


# Current Data Predictions

In [ ]:
X_test, y_test = processing_new_data(config, final_merge_new, truck_data_scaler, truck_data_encoder)

In [ ]:
train = predict_delay(truck_model, X_train, y_train, final_merge)


In [ ]:
test = predict_delay(truck_model, X_test, y_test, final_merge_new)

In [ ]:
from evidently.metric_preset import ClassificationPreset
classification_performance_report = Report(metrics=[ ClassificationPreset()])

classification_performance_report.run(reference_data=train, current_data=test, column_mapping = column_mapping)

classification_performance_report.show(mode='inline')

In [ ]:
def drop_null_values(data, columns_to_drop):
    try:
        # Drop rows with null values in specific columns
        processed_data = data.dropna(subset=columns_to_drop).reset_index(drop=True)
        print("Null values dropped successfully.")

        return processed_data

    except Exception as e:
        # Log the error
        print(f"An error occurred during null value removal: {str(e)}")


def fill_missing_values_with_mode(data, column_name, mode_value):
    try:
        # Fill missing values with mode
        data[column_name] = data[column_name].fillna(mode_value)
        print(f"Missing values in {column_name} filled with mode value.")

    except Exception as e:
        # Log the error
        print(f"An error occurred during missing value imputation: {str(e)}")


def process_categorical_data(data, encoder, encode_columns):
    try:
        # One-Hot Encode
        encoded_features = list(encoder.get_feature_names_out(encode_columns))
        data[encoded_features] = encoder.transform(data[encode_columns])
        print("One-Hot Encoding completed successfully.")


        print("Original categorical features dropped successfully.")

        return data

    except Exception as e:
        # Log the error
        print(f"An error occurred during categorical data processing: {str(e)}")


def scale_data(data, scaler, cts_cols):
    try:
        # Scale separate columns using Standard Scaler
        data[cts_cols] = scaler.transform(data[cts_cols])
        print("Data scaling completed successfully.")

        return data

    except Exception as e:
        # Log the error
        print(f"An error occurred during data scaling: {str(e)}")

def processing_new_data(config, data, scaler, encoder):

    try:
        # Specify
        columns_to_drop_nulls = config['features']['columns_to_drop_null_values']
        encode_columns = config['features']['encode_column_names']
        cts_cols = config['features']['cts_col_names']
        cat_cols = config['features']['cat_col_names']
        target = config['features']['target']

        # Drop null values from specified columns
        data = drop_null_values(data, columns_to_drop_nulls)

        # Fill missing values with mode
        fill_missing_values_with_mode(data, 'load_capacity_pounds', 3000)

        # One-Hot Encode and Drop original categorical features
        data = process_categorical_data(data, encoder, encode_columns)

        # Scale data
        data = scale_data(data, scaler, cts_cols)

        # Save encoder and scaler
        # save_encoder_scaler(encoder, scaler)

        # Extracting features and target variables
        X_test = data[cts_cols + cat_cols]
        y_test = data[target]

        X_test = X_test.drop(encode_columns, axis=1)

        return X_test, y_test

    except Exception as e:
        # Log the error
        print(f"An error occurred while processing the data: {str(e)}")


In [ ]:
def fetch_best_model(config):
    os.environ['WANDB_API_KEY'] = config['wandb']['wandb_api_key']
    PROJECT_NAME = config['wandb']['wandb_project']
    USER_NAME= config['wandb']['wandb_user']
    run = wandb.init(project=PROJECT_NAME)

    # try:
    #     model_artifact = wandb_run.use_artifact(f"{USER_NAME}/{PROJECT_NAME}/XGBoost:latest", type="model")
    #     model_dir = model_artifact.download()
    #     wandb_run.finish()
    #     model_path = os.path.join(model_dir, "xgb-truck-model.pkl")
    #     model = joblib.load(model_dir + "/xgb-truck-model.pkl")
    #     return model
    # except Exception as e:
    #     logger.error(f"An Error occurred while loading the model from wandb: {e}")

    runs = wandb.Api().runs(f"{run.entity}/{PROJECT_NAME}")

    # Initialize variables to keep track of the best F1 score and its corresponding run ID
    best_f1_score = -1
    best_run_id = None

    print(f"Run scores comparison starting")
    # Iterate through runs to find the best F1 score
    for run in runs:
        # Fetch the metrics logged during the run
        run_metrics = run.history()

        # Check if all three F1 scores are logged in the metrics
        if "f1_score_train" in run_metrics.columns and "f1_score_valid" in run_metrics.columns and "f1_score" in run_metrics.columns:
            # Get the last F1 scores for training, validation, and test sets
            f1_score_train = run_metrics['f1_score_train'].max()
            f1_score_valid = run_metrics['f1_score_valid'].max()
            f1_score_test = run_metrics['f1_score'].max()

            # Calculate the average F1 score to prioritize consistency
            avg_f1_score = np.median([f1_score_train , f1_score_valid ,f1_score_test])

            # Update the best F1 score and run ID if a higher F1 score is found
            if avg_f1_score > best_f1_score:
                best_f1_score = avg_f1_score
                best_run_id = run.id

            # print("f1_score_train", f1_score_train)
            # print("f1_score_valid", f1_score_valid)
            # print("f1_score_test", f1_score_test)
            # print("avg_f1_score", avg_f1_score)
            # print("best_f1_score", best_f1_score)

    # Print the best F1 score and its corresponding run ID
    print(f"Best Average F1 Score: {best_f1_score}")
    print(f"Best Run ID: {best_run_id}")

    # Fetch the details of the best run
    best_run = wandb.Api().run(f"{run.entity}/{PROJECT_NAME}/{best_run_id}")

    artifacts = best_run.logged_artifacts()
    best_model = [artifact for artifact in artifacts if artifact.type == 'model'][0]

    artifact_dir = best_model.download()

    # logger.info("artifact_dir: %s", artifact_dir)

    return artifact_dir






